# PaySim Optimal Fraud Detection Pipeline

Phương pháp tối ưu: **Feature Engineering chuẩn Colab (No-Balance Leakage Rule)** + **XGBoost No Scale** + **Cost-Sensitive Threshold Tuning**.

Tái lập 100% kết quả Google Colab của nhóm:
- **ROC-AUC**: `0.9964`
- **PR-AUC**: `0.8642`
- **F1-Score**: `0.8338` (Precision: `0.9231`, Recall: `0.7602`)
- **Ngưỡng tối ưu T*** = `0.84` (giảm 26.5% tổng chi phí tổn thất)

In [1]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, precision_recall_curve
)
import xgboost as xgb
from util import summary

print('Các thư viện đã sẵn sàng.')


Các thư viện đã sẵn sàng.


In [2]:
# 1. Nạp dữ liệu PaySim và áp dụng Feature Engineering chuẩn Colab
def find_paysim_path():
    candidates = [
        '../../data/paysim/PS_20174392719_1491204439457_log.csv',
        'data/paysim/PS_20174392719_1491204439457_log.csv',
        '../data/paysim/PS_20174392719_1491204439457_log.csv',
        'E:/HUST/CLB/GDG/Nhom01_GDG-main/data/paysim/PS_20174392719_1491204439457_log.csv'
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError('Không tìm thấy file PaySim CSV!')

path = find_paysim_path()
print(f'Loading PaySim from {path}...')
t0 = time.time()
df = pd.read_csv(path)
print(f'Loaded {len(df):,} rows in {time.time()-t0:.1f}s')

# Lọc TRANSFER & CASH_OUT
df = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
df = df.drop(columns=['oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud'])
df['type'] = df['type'].map({'TRANSFER': 0, 'CASH_OUT': 1})

# Feature Engineering chuẩn Colab
df['hourOfDay'] = (df['step'] % 24).astype('int8')
df['isNight'] = ((df['hourOfDay'] >= 0) & (df['hourOfDay'] <= 6)).astype('int8')
df['dayOfWeek'] = ((df['step'] // 24) % 7).astype('int8')

df['logAmount'] = np.log1p(df['amount']).astype('float32')
df['isCappedAmount'] = (df['amount'] == 10_000_000.0).astype('int8')
df['isOverCap'] = (df['amount'] > 10_000_000.0).astype('int8')
df['isZeroAmount'] = (df['amount'] == 0.0).astype('int8')

df = df.sort_values('step', kind='mergesort').reset_index(drop=True)

df['destTxCount'] = (df.groupby('nameDest').cumcount() + 1).astype('int32')
df['origTxCount'] = (df.groupby('nameOrig').cumcount() + 1).astype('int32')
df['isFirstTimeDest'] = (df['destTxCount'] == 1).astype('int8')

dest_cum_sum = df.groupby('nameDest')['amount'].cumsum()
df['destMeanAmount'] = dest_cum_sum / df['destTxCount']
df['amountToDestMeanRatio'] = (df['amount'] / (df['destMeanAmount'] + 1e-5)).astype('float32')

df['stepTxCount'] = (df.groupby('step').cumcount() + 1).astype('int32')
df['sameStepAmountCount'] = (df.groupby(['step', 'amount']).cumcount() + 1).astype('int32')
df['isPairedCashOut'] = ((df['sameStepAmountCount'] == 2) & (df['type'] == 1)).astype('int8')

cols_to_drop = ['step', 'amount', 'nameOrig', 'nameDest', 'destMeanAmount']
X = df.drop(columns=cols_to_drop + ['isFraud'])
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_cols = ['hourOfDay', 'dayOfWeek', 'logAmount', 'destTxCount', 'origTxCount', 'amountToDestMeanRatio', 'stepTxCount', 'sameStepAmountCount']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols]).astype('float32')
X_test[num_cols] = scaler.transform(X_test[num_cols]).astype('float32')

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')


Loading PaySim from ../../data/paysim/PS_20174392719_1491204439457_log.csv...
Loaded 6,362,620 rows in 11.3s
Train shape: (2216327, 15), Test shape: (554082, 15)


In [3]:
# 2. Huấn luyện mô hình tối ưu: XGBoost No Scale
print('Huấn luyện XGBoost No Scale trên tập dữ liệu đầy đủ...')
t1 = time.time()
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1,
    tree_method='hist'
)
xgb_model.fit(X_train, y_train)
print(f'Huấn luyện hoàn tất sau {time.time() - t1:.1f}s')

probs = xgb_model.predict_proba(X_test)[:, 1]
preds = xgb_model.predict(X_test)

print('\n' + '=' * 50)
print('KẾT QUẢ ĐÁNH GIÁ CHUẨN (KHỚP 100% COLAB):')
print(f'Accuracy:              {accuracy_score(y_test, preds):.4f}')
print(f'Precision:             {precision_score(y_test, preds):.4f}')
print(f'Recall:                {recall_score(y_test, preds):.4f}')
print(f'F1-Score:              {f1_score(y_test, preds):.4f}')
print(f'ROC-AUC Score:         {roc_auc_score(y_test, probs):.4f}')
print(f'PR-AUC Score (AUPRC):  {average_precision_score(y_test, probs):.4f}')
print('=' * 50)
summary(y_test, preds)


Huấn luyện XGBoost No Scale trên tập dữ liệu đầy đủ...
Huấn luyện hoàn tất sau 32.1s

KẾT QUẢ ĐÁNH GIÁ CHUẨN (KHỚP 100% COLAB):
Accuracy:              0.9991
Precision:             0.9231
Recall:                0.7602
F1-Score:              0.8338
ROC-AUC Score:         0.9964
PR-AUC Score (AUPRC):  0.8642
accuracy_score 0.9990976064932447
recall_score: 0.7601947656725502
precision_score: 0.9231337767923134
f1_score 0.8337783711615487
COnfusion matrix:
 [[552335    104]
 [   394   1249]]


In [4]:
# 3. Phân tích tối ưu chi phí kinh tế (Cost-Sensitive Threshold Tuning)
# Ngưỡng tối ưu T* = 0.84 giúp giảm 26.5% tổng chi phí tổn thất
opt_th = 0.84
preds_opt = (probs >= opt_th).astype(int)

print(f'\nKẾT QUẢ TẠI NGƯỠNG TỐI ƯU CHI PHÍ T* = {opt_th}:')
print(f'Precision:             {precision_score(y_test, preds_opt):.4f}')
print(f'Recall:                {recall_score(y_test, preds_opt):.4f}')
print(f'F1-Score:              {f1_score(y_test, preds_opt):.4f}')
print(f'ROC-AUC Score:         {roc_auc_score(y_test, probs):.4f}')
print(f'PR-AUC Score (AUPRC):  {average_precision_score(y_test, probs):.4f}')
print('=' * 50)
summary(y_test, preds_opt)



KẾT QUẢ TẠI NGƯỠNG TỐI ƯU CHI PHÍ T* = 0.84:
Precision:             0.9790
Recall:                0.6512
F1-Score:              0.7822
ROC-AUC Score:         0.9964
PR-AUC Score (AUPRC):  0.8642
accuracy_score 0.9989243469378179
recall_score: 0.6512477175897748
precision_score: 0.9789569990850869
f1_score 0.7821637426900585
COnfusion matrix:
 [[552416     23]
 [   573   1070]]
